### Data Preprocessing
* Read in bronze data
* Column casting
* Count missing values per column
* Output as delta table



`workspace.default.gold_no_show_features`

In [0]:
from pyspark.sql.types import IntegerType, BooleanType
from pyspark.sql.functions import col
from pyspark.sql.functions import col, when, sum as spark_sum

In [0]:
%python
bronze_df = spark.table("workspace.default.bronze_features")

In [0]:
%skip
display(bronze_df.limit(10))

In [0]:
# List integer and boolean columns - how to convert boolean and integer to double at once?
integer_cols = [
    c.name for c in bronze_df.schema.fields
    if isinstance(c.dataType, (IntegerType, BooleanType))
]

for column in integer_cols:
    bronze_df = bronze_df.withColumn(column, col(column).cast("double"))

In [0]:
bronze_df = bronze_df.withColumn("Showed_up", col("Showed_up").cast("double"))

In [0]:
%skip
# bronze_df.printSchema()
# cast_int_to_double(bronze_df)

In [0]:
''' Prepare data for modelling
    Find columns with empty recoreds
    Remove rows with > 80% empty fields
'''
# Count missing values per column
missing_counts = bronze_df.agg(*[
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in bronze_df.columns
]).first().asDict()

# Display missing value counts as a summary DataFrame
missing_df = spark.createDataFrame(
    [(c, int(v)) for c, v in missing_counts.items()],
    ["column", "missing_count"]
)
display(missing_df.orderBy("missing_count", ascending=False))

In [0]:
silver_df = bronze_df.drop('PatientId', 'AppointmentID')
# display(silver_df)

### Add unique ID

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# Add unique ID column
silver_df = silver_df.withColumn("record_id", monotonically_increasing_id() + 1)

#### Write to delta table
`workspace.default.silver_no_show_features`

In [0]:
silver_df.write.mode("overwrite").saveAsTable("default.silver_no_show_features")

In [0]:
%sql
-- Step 2: Assign Primary Key
ALTER TABLE workspace.default.silver_no_show_features
ADD CONSTRAINT records_pk PRIMARY KEY (record_id);